# 7.3 一个 Softmax 算子的交付过程

## 概述

前两节认识了 Agent 的整体架构和各阶段方法。本节跟随一个真实的行 Softmax 任务，看它怎样从算子要求一步步走向交付。

全程围绕一个问题：

> **算子如何在保证正确性的前提下，达到整次 kernel 耗时的交付目标？**

### 学习目标

- 沿 Agent 工作过程中的实际产物，了解算子实现过程。
- 根据具体测试和性能结果，判断它们能否支持本次要求。
- 指出下一步应继续、返回修复还是补充验证，并说明依据。

[本章导航](README.md) · [上一节](07.02_agent_workflow.ipynb)

概括为：**看懂产出 → 对照目标判断 → 决定下一步。**

本节约 **40 分钟**。

材料说明：本节使用一次完整工作流留下的真实代码、测试和测量数据。

## 1. 把算子需求描述清楚

**本阶段要解决的问题：为了充分发挥算子交付工作流的能力，一份好的恰当的算子需求是至关重要的，你可以以一份需求md文档的形式呈现，或者直接告诉agent其生成算子所需要的信息**

首先我们来看看真实生产场景下，工作流需要什么，本次任务沿用已经学过的稳定行 Softmax：

$$
Y_{i,j}=\frac{\exp(X_{i,j}-m_i)}{\sum_k\exp(X_{i,k}-m_i)},\qquad m_i=\max_kX_{i,k}.
$$

[任务输入](src/workflow_run/course_inputs/task.md) 已经给出了“每行独立计算”的依据：

- 文字要求是“对输入 `X` 的每一行执行 Softmax”。
- 公式中的 $m_i=\max_k X_{i,k}$ 始终固定行下标 $i$，说明最大值只能从第 $i$ 行取得。
- 分母 $\sum_k\exp(X_{i,k}-m_i)$ 也始终固定行下标 $i$，说明归一化不能混入其他行。
- 独立参考使用 `torch.softmax(X.double(), dim=1)`，其中 `dim=1` 表示沿二维输入的列方向计算，每一行分别得到结果。

因此，对任意输出 $Y_{i,j}$，它只能依赖输入的第 $i$ 行。下面再把本次交付的其他约定整理如下：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">类别</th>
<th style="text-align: left; vertical-align: top;">本次约定</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">输入</td>
<td style="text-align: left; vertical-align: top;">NPU 上连续二维 float32；元素有限且绝对值不超过 80</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">shape</td>
<td style="text-align: left; vertical-align: top;">M 为 4～256 之间的 4 的倍数；N 为 128～2048 之间的 64 的倍数</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">输出</td>
<td style="text-align: left; vertical-align: top;">shape、dtype 和 device 与输入一致；内存另行分配</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">范围</td>
<td style="text-align: left; vertical-align: top;">只交付前向计算</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">数值参考</td>
<td style="text-align: left; vertical-align: top;">CPU PyTorch float64 Softmax，再转为 float32</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">精度标准</td>
<td style="text-align: left; vertical-align: top;">逐元素 <code>rtol=1e-4、atol=1e-6</code>；每行和距 1 不超过 <code>1e-5</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">性能案例</td>
<td style="text-align: left; vertical-align: top;">保留全部 <a href="src/workflow_run/course_inputs/cases.json">9 个指定案例</a></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">性能主指标</td>
<td style="text-align: left; vertical-align: top;">同一 case 中，候选与 B0 的整次 kernel <code>Task Duration</code> 之比</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">性能目标</td>
<td style="text-align: left; vertical-align: top;">每项耗时比不超过 1.05；九项几何均值不超过 0.95</td>
</tr>
</tbody>
</table>


**阶段判断**

功能、输入输出、精度和性能主目标已经明确，可以进入设计阶段。下一步要回答：怎样把“每行独立计算”落实到设备任务中？


## 2. 设计：让任务边界符合计算要求

**设计阶段要解决的关键问题：保证 max 和 sum 计算确实在一行内**

计算一行 Softmax 时，需要先查看这一行的全部元素，求出该行的最大值和指数和，然后才能得到这一行的每个输出。

如果把同一行拆给多个任务，各任务还要交换各自算出的局部最大值和局部和，计算过程会更复杂。本例不需要这样拆分，因为最长的一行也能够由一个任务完整处理：

- 最长一行包含 2048 个 float32 元素，占 8 KiB。
- 初始方案需要保存输入、指数中间值和输出，共约 24 KiB。
- 每个 AIV 可使用的 UB 为 248 KiB，空间足够。

因此，本例采用最直接的任务划分：**一个逻辑 block 负责一整行。** 对第 `i` 行来说，这个任务读取 `X[i, :]`，完成该行的最大值、求和与归一化，再写入 `Y[i, :]`。

这里的逻辑 block 可以理解为设计文档规定的算子最小行为单位。

这些内容记录在关键产物[初始设计文档](src/workflow_run/operators/row_softmax/evidence/design/design_v1.md)中，其数据路径可以归纳为：

```text
任务 i 读取 X 的第 i 行
    → 求这一行的最大值
    → 计算 exp(x - max)
    → 求这一行指数值的和
    → 归一化，写入 Y 的第 i 行
```

上面的数据路径是设计阶段提出的方案。由于此时还没有生成代码，这里的检查不是运行测试，而是进行一次**设计审查**：把算子要求逐条对应到设计中的索引、计算范围和资源预算。

以第 `i` 行为例，可以具体检查：

- **核对索引关系**：设计是否明确让第 `i` 个任务读取 `X[i, :]`，并写入 `Y[i, :]`。
- **核对归约范围**：max 和 sum 的计算范围是否始终保留行下标 `i`，没有混入其他行。
- **复算空间预算**：按最长行重新计算输入、中间值和输出所需空间，并与可用 UB 容量比较。

如果设计文档缺少这些信息、前后矛盾或容量算不通，就要先修改设计。

完成这次设计审查后，本例的**逻辑任务划分**就明确了：输入有 M 行，就创建 M 个逻辑 block，每个 block 负责一行。逻辑 block 是程序定义的一份独立计算任务，运行时再把这些任务调度到可用的物理 AIV 上。

例如，设备有 72 个 AIV：

- M=4 时只有 4 个逻辑任务，只需要其中一部分 AIV 执行。
- M=256 时有 256 个逻辑任务，物理 AIV 会分多轮执行，一个 AIV 可能先后处理多个任务。

因此，设计阶段需要确定“创建多少任务、每个任务处理什么数据”，但不需要把任务固定绑定到某一个物理核。具体由哪个 AIV 执行、按什么先后顺序执行，由设备运行时调度。

**阶段判断与下一步**

设计已经给出任务映射、计算顺序和空间预算。下一步进入生成阶段，检查这些选择是否真正写进代码。


## 3. 生成：将设计文档落实为代码

**本阶段要解决的问题：生成代码是否落实了“一行一任务”？**

Agent 依据 `design.md` 生成了 [`row_softmax.py`](src/workflow_run/operators/row_softmax/evidence/generation/initial/row_softmax.py)。落实并非简单地根据 design.md 生成代码，还要检查设计中的关键选择是否真正落实到具体实现中。

如检查任务范围与搬运代码：

```python
with T.Kernel(rows) as bx:
    x = T.alloc_shared((1, cols), "float32")
    # 省略其他局部变量与计算
    T.copy(X[bx : bx + 1, :], x)
    with T.SimdVF():
        T.reduce_max(x, m, dim=1, clear=True)
    # 省略指数计算
    with T.SimdVF():
        T.reduce_sum(e, s, dim=1, clear=True)
    # 省略归一化计算
    T.copy(y, Y[bx : bx + 1, :])
```

按照下面的关系，可以快速判断代码是否落实了设计：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;"><code>design.md</code> 中的选择</th>
<th style="text-align: left; vertical-align: top;">代码中的对应位置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">输入有 M 行，就创建 M 个任务</td>
<td style="text-align: left; vertical-align: top;"><code>T.Kernel(rows)</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">任务 <code>bx</code> 处理第 <code>bx</code> 行</td>
<td style="text-align: left; vertical-align: top;">读取 <code>X[bx : bx + 1, :]</code>，写入 <code>Y[bx : bx + 1, :]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">max 和 sum 只处理当前行</td>
<td style="text-align: left; vertical-align: top;">对当前任务的局部变量 <code>x</code> 和 <code>e</code> 沿列方向归约</td>
</tr>
</tbody>
</table>

这一步是从代码中核对“设计有没有落实”。随后，Agent 还实际运行了 9 个指定案例；它们全部通过，说明已经得到一个能够编译和运行的初始版本。

> **扩展阅读：生成阶段留下的材料**
>
> 第一次阅读这些文件时，可以先看[生成阶段材料阅读说明](src/workflow_run/operators/row_softmax/evidence/generation/README.md)。
>
> - [pytest 原始执行日志](src/workflow_run/operators/row_softmax/evidence/generation/accuracy.log)：查看实际收集和运行了哪些测试，以及最终通过数。
> - [结构化结果](src/workflow_run/operators/row_softmax/evidence/generation/result.json)：查看九个案例的误差指标。
> - [生成 AscendC 源码](src/workflow_run/operators/row_softmax/evidence/generation/generated_source/m64_n512.cpp)：查看一个案例最终生成的设备代码。
> - [源码核对记录](src/workflow_run/operators/row_softmax/evidence/generation/source_check.json)：查看九项源码的哈希和关键后端结构检查。

**阶段判断与下一步**

设计中的任务划分已经落实到代码，初始实现也通过了指定案例，可以进入 ST 检查测试覆盖与可信度。


## 4. ST：用可信测试守住正确性

**本阶段要解决的问题：通过充分的覆盖设计，保障算子高质量交付**

本文选择一个预期容易推导的场景来讲述ST的工作：**验证常数行的数学行为。**

例如，一行有 128 个数且全部为 80：

- 减去本行最大值后，所有数都为 0。
- 取指数后，所有数都为 1。
- 分母为 128，因此每个输出应为 `1/128`。

### 4.1 一条可信测试怎样形成

本轮加入全 0、全 80 和全 −80 的常数输入，并在三个合法 shape 上验证。预期来自公式，不是从 kernel 输出反推答案。

[实际测试代码](src/workflow_run/operators/row_softmax/test_row_softmax.py) 先核对独立 reference：

```python
expected = reference(x)
if pattern in ("zero", "positive80", "negative80"):
    assert torch.equal(expected, torch.full_like(x, 1.0 / shape[1]))
exercise(x, f"{pattern}_{shape}")
```

`exercise` 随后调用实际 NPU 算子，并逐元素比较：

```python
error = (actual.double() - expected_cpu.double()).abs()
allowance = 1e-6 + 1e-4 * expected_cpu.double().abs()
assert (error <= allowance).all()
```

一条测试并不会因为“运行通过”就自动可信。Agent 需要把要求、风险、预期和执行结果连成一条能够核对的链路：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">可信环节</th>
<th style="text-align: left; vertical-align: top;">本例怎样落实</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">找到要求依据</td>
<td style="text-align: left; vertical-align: top;">公式规定每行独立计算；一行元素全部相等时，Softmax 必须得到均匀分布</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">针对风险设计输入</td>
<td style="text-align: left; vertical-align: top;">使用全 0、全 80 和全 −80，覆盖普通值及合法数值范围的两端；再组合 <code>(4,128)</code>、<code>(68,192)</code> 和 <code>(256,2048)</code>，覆盖最小、中间和最大规模</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">建立可靠预期</td>
<td style="text-align: left; vertical-align: top;">先按公式推出每个元素应为 <code>1/N</code>，再用 CPU PyTorch float64 reference 独立计算；两种来源相互核对</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">使用有效断言</td>
<td style="text-align: left; vertical-align: top;">对实际输出的每个元素检查 <code>rtol=1e-4、atol=1e-6</code>，同时检查结果有限且非负、每行和接近 1、shape/dtype/device 正确、输出另行分配且输入未被修改</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">留下执行证据</td>
<td style="text-align: left; vertical-align: top;">保存实际收集的 pytest nodeid、通过数、返回码和误差指标</td>
</tr>
</tbody>
</table>

### 4.2 覆盖设计

刚才的常数行用例就是一次覆盖设计：Agent 从算子要求中选出一个值得验证的行为，再为它确定输入、预期结果和有效断言。一个个这样的覆盖设计共同减少未经验证的区域，从而提升算子高质量交付的可信程度。

这个例子自然涉及了多个覆盖点：

- **功能与精度**：验证相同行元素是否得到均匀分布，并与独立 reference 逐元素比较。
- **边界**：使用 0、+80、−80，以及最小、中间和最大 shape，检查数值与规模边界。
- **状态修改、布局与接口**：确认输入没有被改写，输出另行分配，并检查输出的 shape、dtype、device 和连续性。
- **后端与分支、执行证据**：通过公开接口在 NPU 上运行实际算子，保存 pytest nodeid 和运行结果。这个例子证明实际 NPU 路径被执行。

还有三个覆盖点没有在这个例子中直接体现：

- **异常拒绝**：常数行使用的是合法输入，还要另行验证非法 shape、dtype、数值、布局、设备和 builder 参数能否按约定报错。
- **梯度**：本次需求只交付前向计算，因此不适用。
- **随机性**：算子没有随机计算，因此不适用。

“不适用”也是覆盖设计的判断结果，但必须能从算子要求中找到依据。最终覆盖结论可查看 [ST 验收报告](src/workflow_run/operators/row_softmax/st_report.md)。

### 4.3 确认测试确实执行

- 完整 ST：**90 项通过、0 项失败、0 项跳过**。
- [Level 1 记录](src/workflow_run/operators/row_softmax/evidence/st/level1.json) 保存了实际执行信息。

**阶段判断与下一步**

当前阶段完成了ST检查，可以冻结为性能基线 B0。后面的调优必须先保持这些行为。


## 5. 调优：提升算子性能

**本阶段要解决的问题：提升算子性能**

### 5.1 从事实形成候选

ST 通过后，Agent 将当前正确实现冻结为性能基线 B0，进入调优阶段。Agent 先分析源码和生成代码中的计算、数据搬运及中间结果存放方式，并形成[源码成本分析](src/workflow_run/operators/row_softmax_20260916_002620/semantic_cost_model.md)。

本例使用 `msprof` 采集 B0 在九个指定 case 上的设备侧性能数据，记录每次 kernel 的 `Task Duration`。`msprof` 提供实际测量结果，Agent 再将这些结果与源码中的计算和数据搬运过程对照，寻找可能的优化方向。

将源码分析与 `msprof` 采集结果对照后，Agent 得到两个可检查的事实：

- 长行的 max 和 sum 都要先分段归约，再合并分段结果。
- 指数中间值写入 UB 后，求和与归一化还要再次读取。

#### C1：减少分段归约次数

针对第一个事实，Agent 提出了候选 C1。原实现对每个数据段分别归约，再合并多个分段结果；C1 先以向量形式累积各段数据，处理完整行后再进行一次归约，希望减少重复的归约与合并开销：

<img alt="同一行的求和怎样调整计算组织" src="images/07.03_reduction.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

下面两个关键片段可以直观看出 B0 与 C1 的区别。

B0 先计算并保存整行指数，再单独调用归约：

```python
# 第一步：计算本行每个元素的指数，并保存到 UB 中的 e
with T.SimdVF():
    for i, j in T.Parallel(1, cols):
        e[i, j] = T.exp(x[i, j] - m[i])
# 第二步：重新读取整行 e，归约得到分母 s
with T.SimdVF():
    T.reduce_sum(e, s, dim=1, clear=True)
```

C1 把指数计算与逐 lane 累加放入同一个 SimdVF，最后只做一次水平求和：

```python
# 用一个 64-lane 向量寄存器保存分段累加结果
lane_sum = S.vdup(0.0, "float32", mask)
for k in T.serial(cols // 64):
    # 每次读取一段 64 个元素，并计算指数
    value = S.vexp(S.vsub(S.vld(x[0, k * 64]), row_max, mask), mask)
    # 保存指数，供后面的归一化再次使用
    S.vsts(e[0, k * 64], value, mask)
    # 各段相同 lane 的值先在向量中累加
    lane_sum = S.vadd(lane_sum, value, mask)
# 同步指令：等待前面的向量写入（VST）完成，再执行后面的向量读取（VLD）
S.mem_bar("VST_VLD")
# 将 64 个 lane 的部分和归约成整行分母
row_sum = S.vdupv(S.vcadd(lane_sum, mask), mask)
for k in T.serial(cols // 64):
    # 重新读取保存的指数，除以分母得到输出
    result = S.vdiv(S.vld(e[0, k * 64]), row_sum, mask)
```
但后续的性能测量显示，九个案例的整次 kernel `Task Duration` 几何均值变为 B0 的 `1.293` 倍，整体耗时增加约 29.3%。这证明 C1 没有达到性能目标。Agent 则会将 C1 记录为拒绝，回到 B0 继续寻找其他候选。

agent分析的原因是：虽然C1 成功减少了水平归约次数，说明了优化设想在计算结构上能够落实。但 e 需写入 UB 并在归一化时重新读取，引入了新的写后读同步需求，这种新增成本可能覆盖了归约可能带来的收益，最终导致整体性能回退。Agent 因而没有采用完整的 C1，但在后续 C4中保留了它的归约组织，并继续消除 e 的 UB 存取和同步需求。


#### C4：减少指数中间值的 UB 往返

C1 减少了分段归约次数，但仍要把指数中间值 `E` 写入 UB，并在生成输出时读回。C4 保留 C1 的逐 lane 累加方式，同时不再保存整行 `E`：它在生成输出时**重新计算指数**，再乘以分母的倒数。

<img alt="减少指数中间值的 UB 往返" src="images/07.03_c4_recompute.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

两种方案计算分母的都是通过逐lane累加的方式，所以下面代码的呈现省略这部分，只比较生成输出时怎样取得 `E`。此时 C1 已得到 `row_sum`，C4 已得到 `reciprocal = 1 / row_sum`。

C1 等待 UB 写入完成后，读回第一遍保存的 `E`：

```python
# C1：此时指数 E 已写入 UB
S.mem_bar("VST_VLD") # 等待前面的 VST 完成，再执行后续 VLD
for k in T.serial(cols // 64):
    value = S.vld(e[0, k * 64])  # 从 UB 读回 E
    result = S.vdiv(value, row_sum, mask)
    S.vsts(y[0, k * 64], result, mask)
```

C4 不保存 `E`，生成输出时从 `x` 重新计算：

```python
# C4：输出阶段再次计算 E
for k in T.serial(cols // 64):
    value = S.vexp(S.vsub(S.vld(x[0, k * 64]), row_max, mask), mask)
    result = S.vmul(value, reciprocal, mask)
    S.vsts(y[0, k * 64], result, mask)
```

C1 避免了重复计算指数，但需要写入 `E`、等待写后读同步，并在输出阶段从 UB 读回。C4 去掉了这些操作，代价是重新读取 `x` 并再计算一次指数。接下来我们看实测结果，C4是否有优势。

### 5.2 用实测结果确定最终方案

实际调优过程中，Agent 形成并验证了从 C1 到后续编号的多个候选版本。本节只选取两个有代表性的候选：C1 通过精度测试后因性能回退被拒绝，C4 通过正确性和性能复验后晋升为最终方案。接下来比较 C4 与性能基线 B0。

下图使用已保存的同批 `Task Duration` 数据，比较 C4 与 B0。每根柱都是“C4 中位数 ÷ B0 中位数”，越小越好。

<img alt="九个案例的 kernel duration 复验" src="images/07.03_workflow_performance.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

最终结果是：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">检查</th>
<th style="text-align: left; vertical-align: top;">结果</th>
<th style="text-align: left; vertical-align: top;">门槛</th>
<th style="text-align: left; vertical-align: top;">判断</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">九项几何均值</td>
<td style="text-align: left; vertical-align: top;">0.93322</td>
<td style="text-align: left; vertical-align: top;">≤ 0.95</td>
<td style="text-align: left; vertical-align: top;">通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">最差单项 <code>(64,128)</code></td>
<td style="text-align: left; vertical-align: top;">1.00792</td>
<td style="text-align: left; vertical-align: top;">≤ 1.05</td>
<td style="text-align: left; vertical-align: top;">通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">达到单项上限的案例</td>
<td style="text-align: left; vertical-align: top;">9/9</td>
<td style="text-align: left; vertical-align: top;">9/9</td>
<td style="text-align: left; vertical-align: top;">通过</td>
</tr>
</tbody>
</table>

从逐案例结果可以看出，收益随行长增加而变得明显：`N=128` 的三个案例基本持平，`N=512` 的两个案例缩短约 4%～5.5%，`N=2048` 的三个案例缩短约 11.7%～17.8%。一行有 `N/64` 个向量分段，因此 `N=128` 只需处理 2 段，而 `N=2048` 需要处理 32 段。

可以看到：**行越长，C4 省去的 `E` 写入、读回和逐段除法越多，固定开销在整次执行中的占比也越小**。虽然 C4 增加了指数重算，但本次结果表明：**在长行上，减少 UB 往返并用乘法替代逐段除法所带来的收益，超过了重新读取输入和重算指数的额外开销。**

[课程性能汇总](src/workflow_run/operators/row_softmax_20260916_002620/comparisons/course_kernel_duration_summary.json) 给出逐案例的 B0、C4 中位数和比值。


**阶段判断与下一步**

C4 保持正确性，九项 kernel duration 也满足既定门槛，可以进入最终 Review。这里证明的是“达到本次交付要求”，不代表已经穷尽所有可能实现。


## 6. 交付：确认最终实现满足交付要求

**本阶段对算子进行交付前的最终检查**

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">核对项</th>
<th style="text-align: left; vertical-align: top;">最终证据</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">最终实现</td>
<td style="text-align: left; vertical-align: top;">保留 C4</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">正确性</td>
<td style="text-align: left; vertical-align: top;">完整 90 项 ST 通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">性能口径</td>
<td style="text-align: left; vertical-align: top;">整次 kernel <code>Task Duration</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">整体性能</td>
<td style="text-align: left; vertical-align: top;">九项几何均值比 0.93322，满足不超过 0.95</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">单项性能</td>
<td style="text-align: left; vertical-align: top;">最差 <code>(64,128)</code> 为 1.00792，九项均不超过 1.05</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">代码 Review</td>
<td style="text-align: left; vertical-align: top;">格式与 lint 检查通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">证据对应</td>
<td style="text-align: left; vertical-align: top;">交付副本重新通过 90 项 ST；生成 kernel 与性能采集版本一致</td>
</tr>
</tbody>
</table>

[课程调优报告](src/workflow_run/operators/row_softmax_20260916_002620/flash-report.md) 汇总了候选与 kernel duration 结论。[最终设计](src/workflow_run/operators/row_softmax_20260916_002620/delivery_reviewed/design_final.md) 说明了 C4 相比初版的变化，算子、pytest 和检查记录集中在[交付导览](src/workflow_run/README.md)。

## 小结

回过头看，本例从把需求说清楚开始：Softmax 要逐行独立计算，输入范围、精度要求和性能目标都要提前明确。Agent 据此选择“一行一个任务”的设计，并说明数据怎样读取、计算和写回，以及局部空间是否够用。

设计落实成代码后，Agent 会检查读取、归约和写回是否符合原来的约定。接着通过 ST 补齐测试覆盖，用可靠的预期和断言检验结果；发现问题时，再回到实现中修正。正确性通过后，Agent 才结合源码与 `msprof` 数据提出优化候选，并用正确性结果和整次 kernel 耗时决定采用还是淘汰。

最后，Review 会再次核对最终代码、测试结果和性能证据是否对应同一版本。只有这些检查都通过，算子才算完成本次交付。整个过程中，每个阶段都要拿出可检查的执行证据，并根据结果判断下一步该继续、修正还是回退。

## 课后练习

具备课程环境后，请参考[完整流程练习](src/exercise_task.md)，在独立仓库中尝试运行一次从需求与设计、代码生成、ST、性能调优到 Review 和交付的完整 Agent 流程。保存各阶段的产物和执行证据，并说明每一步为什么可以继续、修正或回退。


[上一节：Skills 编排原理](07.02_agent_workflow.ipynb) · [返回章节目录](README.md)